In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# Define weather api endpoint
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"

# Set query parameters for coordinate location
# Latitude and longitude for a sample metropolitan area
API_PARAMS = {
    "latitude": 52.52,
    "longitude": 13.41,
    "hourly": "temperature_2m,rain",
    "past_days": 7,
    "forecast_days": 1
}

# Define output storage path
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
def fetch_weather_data(url, params):
    try:
        # Send get request to open meteo api
        response = requests.get(url, params=params, timeout=15)
        
        # Check connection status
        if response.status_code != 200:
            print(f"API connection failed with status: {response.status_code}")
            return None
            
        # Parse json response
        data = response.json()
        
        # Extract hourly data series
        hourly_data = data.get("hourly", {})
        timestamps = hourly_data.get("time", [])
        temperatures = hourly_data.get("temperature_2m", [])
        rain_levels = hourly_data.get("rain", [])
        
        # Build records list
        records = []
        for i in range(len(timestamps)):
            records.append({
                "raw_timestamp": timestamps[i],
                "raw_temp": temperatures[i],
                "raw_rain": rain_levels[i],
                "api_fetched_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            
        return records
        
    except Exception as e:
        # Print network error message
        print(f"Error connecting to weather API: {e}")
        return None
        

In [3]:
# Execute api data collection
weather_records = fetch_weather_data(WEATHER_API_URL, API_PARAMS)

# Ingest records into dataframe
if weather_records:
    df_weather = pd.DataFrame(weather_records)
    print(f"Total weather data points ingested: {len(df_weather)}")
    print(df_weather.head())
else:
    print("Pipeline failed to collect data")

Total weather data points ingested: 192
      raw_timestamp  raw_temp  raw_rain       api_fetched_at
0  2026-06-01T00:00      14.4       0.0  2026-06-08 17:07:00
1  2026-06-01T01:00      14.1       0.0  2026-06-08 17:07:00
2  2026-06-01T02:00      14.1       0.0  2026-06-08 17:07:00
3  2026-06-01T03:00      13.9       0.0  2026-06-08 17:07:00
4  2026-06-01T04:00      13.3       0.0  2026-06-08 17:07:00


In [4]:
def preprocess_weather_pipeline(dataframe):
    # Drop rows with missing values
    dataframe = dataframe.dropna(subset=["raw_timestamp", "raw_temp", "raw_rain"])
    
    # Create clean dataframe copy
    clean_df = dataframe.copy()
    
    # Convert string timestamps to datetime objects
    clean_df["timestamp"] = pd.to_datetime(clean_df["raw_timestamp"])
    
    # Ensure numeric data types
    clean_df["temperature"] = clean_df["raw_temp"].astype(float)
    clean_df["rainfall"] = clean_df["raw_rain"].astype(float)
    
    # Drop original raw columns
    clean_df = clean_df.drop(columns=["raw_timestamp", "raw_temp", "raw_rain"])
    
    return clean_df

In [5]:
# Execute preprocessing pipeline
df_weather_clean = preprocess_weather_pipeline(df_weather)

# Print preprocessed dataframe status
print("Weather data preprocessing completed successfully")
print(df_weather_clean.dtypes)
df_weather_clean[["timestamp", "temperature", "rainfall"]].head()

Weather data preprocessing completed successfully
api_fetched_at            object
timestamp         datetime64[ns]
temperature              float64
rainfall                 float64
dtype: object


,timestamp,temperature,rainfall
0,2026-06-01 00:00:00,14.4,0.0
1,2026-06-01 01:00:00,14.1,0.0
2,2026-06-01 02:00:00,14.1,0.0
3,2026-06-01 03:00:00,13.9,0.0
4,2026-06-01 04:00:00,13.3,0.0


In [6]:
def assign_weather_condition(rain_val):
    # Categorize condition by rainfall volume
    if rain_val == 0.0:
        return "Clear"
    elif rain_val <= 2.0:
        return "Light Rain"
    else:
        return "Heavy Rain"

def transform_weather_features(dataframe):
    # Extract hour from datetime object
    dataframe["hour"] = dataframe["timestamp"].dt.hour
    
    # Generate categorical weather conditions
    dataframe["condition"] = dataframe["rainfall"].apply(assign_weather_condition)
    
    return dataframe

In [7]:
# Execute feature engineering pipeline
df_weather_transformed = transform_weather_features(df_weather_clean)

# Print current transformation status
print("Weather feature engineering completed successfully")
df_weather_transformed[["timestamp", "temperature", "rainfall", "hour", "condition"]].head()

Weather feature engineering completed successfully


,timestamp,temperature,rainfall,hour,condition
0,2026-06-01 00:00:00,14.4,0.0,0,Clear
1,2026-06-01 01:00:00,14.1,0.0,1,Clear
2,2026-06-01 02:00:00,14.1,0.0,2,Clear
3,2026-06-01 03:00:00,13.9,0.0,3,Clear
4,2026-06-01 04:00:00,13.3,0.0,4,Clear


In [ ]:
def compute_weather_patterns(dataframe):
    # Group by weather condition
    summary = dataframe.groupby("condition").agg(
        average_temperature=("temperature", "mean"),
        total_rainfall=("rainfall", "sum"),
        hours_recorded=("timestamp", "count")
    ).reset_index()
    
    return summary

def plot_weather_trends(dataframe):
    # Initialize figure size
    fig, ax1 = plt.subplots(figsize=(14, 7))
    
    # Plot temperature on primary y axis
    sns.lineplot(data=dataframe, x="timestamp", y="temperature", ax=ax1, color="#ef4444", label="Temperature (°C)", linewidth=2)
    ax1.set_title("7-Day Weather Trends: Temperature vs Rainfall", fontsize=14)
    ax1.set_xlabel("Date and Time")
    ax1.set_ylabel("Temperature (°C)", color="#ef4444")
    ax1.tick_params(axis="y", labelcolor="#ef4444")
    
    # Create secondary y axis for rainfall
    ax2 = ax1.twinx()
    sns.lineplot(data=dataframe, x="timestamp", y="rainfall", ax=ax2, color="#2563eb", label="Rainfall (mm)", linewidth=1.5, linestyle="--")
    ax2.set_ylabel("Rainfall (mm)", color="#2563eb")
    ax2.tick_params(axis="y", labelcolor="#2563eb")
    
    # Combine legends from both axes
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    
    # Optimize layout format
    plt.tight_layout()
    plt.show()

In [ ]:
# Force inline notebook plotting
%matplotlib inline

# Set chart design theme
sns.set_theme(style="whitegrid")

# Compute data mining pattern summaries
weather_summary = compute_weather_patterns(df_weather_transformed)
print("Weather Patterns by Condition Category:")
print(weather_summary)

# Generate time series dual chart
plot_weather_trends(df_weather_transformed)

In [ ]:
import json

def deploy_weather_storage(dataframe, summary_df, output_path):
    # Define file path locations
    csv_filename = os.path.join(output_path, "weather_processed_data.csv")
    json_filename = os.path.join(output_path, "weather_pattern_summary.json")
    
    # Save transformed dataframe to disk
    dataframe.to_csv(csv_filename, index=False)
    
    # Convert summary records to dictionary
    summary_dict = summary_df.to_dict(orient="records")
    
    # Build metadata architecture
    final_report = {
        "export_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_hours_analyzed": len(dataframe),
        "pattern_insights": summary_dict
    }
    
    # Save json summary document
    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=4)
        
    print("Weather storage pipeline deployment completed successfully")

# Execute final weather storage
deploy_weather_storage(df_weather_transformed, weather_summary, OUTPUT_DIR)